# 03 – Walk-forward backtest and metrics (tasks 7–8)

Runs `src/backtest.py` for the three portfolios of the plan and checks the result.
Timing and cost conventions: module docstring and `docs/open_decisions.md`.

In [ ]:
# Setup
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "data.py").exists())
sys.path.insert(0, str(ROOT))
from src import backtest, data
from src.optimizer import BENCH, check_rules

TICKERS = data.TICKERS
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", None)

## 1. Short sample first (2016)

In [ ]:
daily, monthly = backtest.run_backtest("full", first="2015-12", last="2016-11")
print(f"Sessions: {daily.index[0].date()} -> {daily.index[-1].date()} | {len(daily)}")
print(monthly[["traded", "cost", "te_ex_ante", "relaxed_turnover_cap"]].round(4))

## 2. Full run: benchmark, BL with TE budget only, BL under full mandate (saved to `data/results/`)

In [ ]:
net, gross, log = backtest.run_all()
print(f"Evaluation: {net.index[0].date()} -> {net.index[-1].date()} | {len(net)} sessions")
print("Files:", sorted(p.name for p in backtest.RESULTS_DIR.glob("*.csv")))

## 3. Checks

In [ ]:
W = log.set_index(["portfolio", "month"])[[f"w_{t}" for t in TICKERS]].set_axis(TICKERS, axis=1)
rules = {"benchmark": "full", "bl_te_only": "long_only", "bl_full": "full"}
viol = sum(bool(check_rules(x, rules[p])) for (p, _), x in W.iterrows())
print(f"Weights sum to 1: max error {(W.sum(axis=1) - 1).abs().max():.1e} | rule violations: {viol}")

relaxed = log.dropna(subset=["relaxed_turnover_cap"])
print("Turnover cap relaxed:", relaxed[["portfolio", "month", "relaxed_turnover_cap"]].to_dict("records"))

cost_gap = ((1 + gross).prod() / (1 + net).prod() - 1)
print("Cumulative cost drag:", cost_gap.round(5).to_dict())

# Benchmark vs static weights on monthly returns: gap = intra-month drift + next-day rebalance only
_, _, mon = data.load_processed()
static = mon.loc["2016-01":"2026-10", TICKERS] @ BENCH
realized = (1 + net["benchmark"]).groupby(net.index.to_period("M")).prod() - 1
print(f"Benchmark vs static monthly: mean |gap| {(realized - static).abs().mean():.2e}, "
      f"max {(realized - static).abs().max():.2e} ({(realized - static).abs().idxmax()})")

## 4. Metrics (task 8)

`src/metrics.py`, saved to `data/results/metrics.csv` and `metrics_by_year.csv` for the app.
Net of 5 bp costs; annualized with 252 sessions; ES as a positive daily loss.

In [ ]:
from src import metrics

table, yearly = metrics.compute_all()
print(table.T.round(4))

Realized TE and active return by calendar year (is the 2% budget respected out of sample?)

In [ ]:
print(yearly.round(4))